## Data Extraction

In [1]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/Medical_book.pdf")
extracted_data = loader.load()

# for i, doc in enumerate(extracted_data):
#     if "acne" in doc.page_content.lower():
#         print(f"\n--- Page {i + 1} ---")
#         print(doc.page_content[:3000])

In [3]:
# extracted_data

In [4]:
len(extracted_data)

637

In [5]:
from typing import List
from langchain.schema import Document

# def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
#     """
#     Given a list of document objects, return a new list of document objects containing only 'source' in metdata and the original page_content
#     """

#     minimal_docs: List[Document] = []

#     for doc in docs:
#         src = doc.metadata.get("source")

#         minimal_docs.append(
#             Document(
#                 page_content=doc.page_content,
#                 metadata={"source": src}
#             )
#         )

#     return minimal_docs

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    return [
        Document(
            page_content=doc.page_content,
            metadata={"source": doc.metadata.get("source")}
        )
        for doc in docs
        if doc.page_content.strip()
    ]

minimal_docs = filter_to_minimal_docs(extracted_data)

print("Total extracted pages:", len(extracted_data))
print("Non-empty pages:", len(minimal_docs))

Total extracted pages: 637
Non-empty pages: 636


In [6]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [7]:
# minimal_docs

## Chunking (Split document into smaller chunks)

In [8]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100,
        separators=["\n\n", "\n", " ", ""]
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [9]:
text_chunk = text_split(minimal_docs)
print(f"Number of Chunks: {len(text_chunk)}")

Number of Chunks: 6600


In [34]:
# for i, doc in enumerate(text_chunk):
#     text = doc.page_content.lower()

#     if "acne" in text and len(text) > 200:
#         print(f"\n--- Chunk {i} ---")
#         print(text[:1000])


--- Chunk 203 ---
• dental and throat abscesses by mouth flora
• lung abscesses by normal airway flora, pneumonia
germs, or tuberculosis
• abdominal and anal abscesses by normal bowel flora
specific types of abscesses
listed below are some of the more common and
important abscesses.
• carbuncles and other boils. skin oil glands (sebaceous
glands) on the back or the back of the neck are the ones
usually infected. the most common germ involved is
staphylococcus aureus . acne is a similar condition of

--- Chunk 204 ---
staphylococcus aureus . acne is a similar condition of
sebaceous glands on the face and back.
• pilonidal abscess. many people have as a birth defect a
tiny opening in the skin just above the anus. fecal bac-
teria can enter this opening, causing an infection and
subsequent abscess.
gale encyclopedia of medicine 2 13
abcess
an amoebic abscess caused by entameoba histolytica.
(phototake nyc. reproduced by permission.)
gem - 0001 to 0432 - a  10/22/03 1:41 pm  page 13

--- 

## Embedding Model

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embedding():
    """
    Download and return the HuggingFace embeddings model
    """

    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name = model_name
    )

    return embeddings

embedding = download_embedding()

C:\Users\DELL\AppData\Local\Temp\ipykernel_10496\2552162825.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [11]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [12]:
vector = embedding.embed_query("Hello World")

In [13]:
print(f"Length of Vector: {len(vector)}")

Length of Vector: 384


## Pinecone Vector DB

In [14]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [15]:
MEDICALBOT_PINECONE_API_KEY = os.getenv("MEDICALBOT_PINECONE_API_KEY")
GOOGLE_GEN_AI = os.getenv("GOOGLE_GEN_AI")

os.environ["MEDICALBOT_PINECONE_API_KEY"] = MEDICALBOT_PINECONE_API_KEY
os.environ["GOOGLE_GEN_AI"] = GOOGLE_GEN_AI

In [16]:
from pinecone import Pinecone 

pinecone_api_key = MEDICALBOT_PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_api_key)

In [17]:
pc

In [18]:
from pinecone import ServerlessSpec 

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

#### Run the cell only once

In [19]:
# from langchain_pinecone import PineconeVectorStore

# docsearch = PineconeVectorStore.from_documents(
#     documents=text_chunk,
#     embedding=embedding,
#     index_name=index_name
# )

Loading Existing Index

In [20]:
from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

Add More Data to the existing Pinecone index

In [21]:
dummyDoc = Document(
    page_content="Hi this is Ninad More, a software engineer at Barclays",
    metadata={"source": "LinkedIn"}
)

In [40]:
# index = pc.Index(index_name)
# print(index.describe_index_stats())

In [41]:
# query = "Acne is a common skin disease characterized by pimples"

# results = docsearch.similarity_search(query, k=5)

# for i, doc in enumerate(results, 1):
#     print(f"\n--- Result {i} ---")
#     print(doc.page_content)
#     print("Metadata:", doc.metadata)

In [43]:
# from collections import Counter

# chunk_texts = [doc.page_content.strip() for doc in text_chunk]
# counts = Counter(chunk_texts)

# print("Total chunks:", len(text_chunk))
# print("Unique chunks:", len(counts))
# print("Duplicate chunks:", sum(c - 1 for c in counts.values() if c > 1))

# for chunk, count in counts.most_common(10):
#     if count > 1:
#         print(f"\nRepeated {count} times:")
#         print(chunk[:250])

In [42]:
# print(pc.Index(index_name).describe_index_stats())

In [22]:
docsearch.add_documents(documents=[dummyDoc])

['24af19d8-95e6-4065-a3c3-5f3af7d24229']

In [23]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [24]:
retrieved_docs = retriever.invoke("What is Acne?")

In [25]:
retrieved_docs

[Document(id='3a00a4cf-960e-4500-abeb-4d682bc244b7', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='8bbaa41f-3a40-40fb-9dc8-ef557fc62979', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='c3475303-bec2-4ab3-99ea-6ffc9426a531', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26')]

In [26]:
from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=GOOGLE_GEN_AI,
    temperature=0
)

D:\Anaconda\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.13). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
D:\Anaconda\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.13). Google will not post any further updates to google.ai.generativelanguage_v1beta supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.ai.generativelanguage_v1beta.
  warnings.warn(message, FutureWarning)


In [27]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [28]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to asnwer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use the three sentences maximum and keep the "
    "answer consice."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [29]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [30]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

Based on the provided text, acromegaly is a disorder in which the abnormal release of a chemical from the pituitary gland in the brain causes increased growth in bone and soft tissue. The provided text does not contain information to define gigantism.


In [31]:
response = rag_chain.invoke({"input": "What is Acne?"})
print(response["answer"])

I do not know the answer based on the provided context, as it does not contain a definition or description of acne.


In [32]:
response = rag_chain.invoke({"input": "What is Ninad?"})
print(response["answer"])

I do not know the answer based on the provided context.


In [33]:
response = rag_chain.invoke({"input": "How to treat Acne?"})
print(response["answer"])

To manage acne, avoid abrasive cleansers, use noncomedogenic makeup and moisturizers, and do not pick or squeeze blemishes. You should also shampoo frequently, keep hair off your face, reduce stress, and eat a well-balanced diet while avoiding foods that trigger flare-ups. Additionally, unless instructed otherwise, dry pimples may be given limited sun exposure.
